In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Generate date range from 1940-01-01 to 2030-12-31
start_date = "1940-01-01"
end_date = "2030-12-31"

df = (
    spark.sql(f"SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as date_array")
    .selectExpr("explode(date_array) as date")
)

# Build the calendar table with all standard columns
calendar_df = df.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date"),
    F.year("date").alias("year"),
    F.month("date").alias("month"),
    F.dayofmonth("date").alias("day"),
    F.dayofweek("date").alias("day_of_week"),          # 1=Sunday, 7=Saturday
    F.dayofyear("date").alias("day_of_year"),
    F.weekofyear("date").alias("week_of_year"),
    F.quarter("date").alias("quarter"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.date_format("date", "EEE").alias("day_name_short"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_name_short"),
    F.date_format("date", "yyyy-MM").alias("year_month"),
    F.date_format("date", "yyyyMM").cast("int").alias("year_month_key"),
    F.trunc("date", "month").alias("first_day_of_month"),
    F.last_day("date").alias("last_day_of_month"),
    F.trunc("date", "quarter").alias("first_day_of_quarter"),
    F.trunc("date", "year").alias("first_day_of_year"),
    F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    F.when(F.dayofweek("date").isin(1, 7), False).otherwise(True).alias("is_weekday"),
    F.concat(F.lit("Q"), F.quarter("date")).alias("quarter_name"),
    F.concat(F.year("date"), F.lit("-Q"), F.quarter("date")).alias("year_quarter"),
    F.ceil(F.month("date") / 6).cast("int").alias("semester"),
    F.concat(F.year("date"), F.lit("-H"), F.ceil(F.month("date") / 6).cast("int")).alias("year_semester"),
)

# Write the calendar table to Unity Catalog
calendar_df.write.format("delta").mode("overwrite").saveAsTable("adb_caresync.silver.calendar")

print(f"Calendar table is created successfully with {calendar_df.count()} rows.")
display(spark.table("adb_caresync.silver.calendar").limit(10))